In [0]:
%python
from datetime import date, timedelta
import sys
import os

# Add both parent directory and Classes directory to path for imports
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
classes_dir = os.path.join(parent_dir, 'Classes')

for path in [parent_dir, classes_dir]:
    if path not in sys.path:
        sys.path.insert(0, path)

from Classes.fact_generation import DAILY_BASE_ROWS, bronzes_from_frames, generate_day

try:
    dbutils.widgets.text('TARGET_DATE', '', 'Target date (YYYY-MM-DD, blank = yesterday)')
    dbutils.widgets.text('SCALE_FACTOR', '', 'Scale factor (blank = 1.0)')
    _raw_date = dbutils.widgets.get('TARGET_DATE').strip()
    _raw_scale = dbutils.widgets.get('SCALE_FACTOR').strip()
except NameError:  # running outside Databricks
    _raw_date, _raw_scale = '', ''
TARGET_DATE = date.fromisoformat(_raw_date) if _raw_date else date.today() - timedelta(days=1)
SCALE_FACTOR = float(_raw_scale) if _raw_scale else 1.0  # 1.0 = ~100K rows/day

CATALOG = 'car_workshop'
FACT_OUTPUT_DIR = f'/Volumes/{CATALOG}/bronze/landing'

_est = sum(max(int(n * SCALE_FACTOR), 1) for n in DAILY_BASE_ROWS.values())
print(f'TARGET_DATE  = {TARGET_DATE}')
print(f'SCALE_FACTOR = {SCALE_FACTOR}  (~{_est:,} rows + schedules)')
print(f'OUTPUT       = {FACT_OUTPUT_DIR}')

In [0]:
%python
# Define your date range
START_DATE = date(2026, 8, 1)   # Modify this
END_DATE = date(2026, 9, 30)    # Modify this

# Generate list of dates
date_list = []
current = START_DATE
while current <= END_DATE:
    date_list.append(current)
    current += timedelta(days=1)

print(f'Generated {len(date_list)} dates from {START_DATE} to {END_DATE}')
print(f'First: {date_list[0]}, Last: {date_list[-1]}')
print(f'\nExample iteration:')
print(f'for target_date in date_list:')
print(f'    # process target_date')

In [0]:
%python
def column(table, col):
    return spark.table(f'{CATALOG}.bronze.{table}').select(col).toPandas()[col].to_numpy()


# raises RuntimeError if any bronze is empty - run initial_bronzes + bronze ingestion first
bronzes = bronzes_from_frames(
    locations=spark.table(f'{CATALOG}.bronze.locations').select('location_id', 'type').toPandas(),
    employees=(spark.table(f'{CATALOG}.bronze.employees')
               .select('employee_id', 'position', 'location_id').toPandas()),
    customer_ids=column('customers', 'customer_id'),
    vehicle_ids=column('vehicles', 'vehicle_id'),
    product_ids=column('products', 'product_id'),
    service_ids=column('services', 'service_id'),
    supplier_ids=column('suppliers', 'supplier_id'),
)
print(f"bronzes loaded: {len(bronzes['customer_ids']):,} customers, "
      f"{len(bronzes['vehicle_ids']):,} vehicles, {len(bronzes['product_ids']):,} products, "
      f"{len(bronzes['employee_ids']):,} employees")

In [0]:
%python
import time

import pandas as pd

for TARGET_DATE in date_list:
    print(f'\nProcessing {TARGET_DATE}...')
    _t0 = time.time()
    stats = generate_day(TARGET_DATE, SCALE_FACTOR, FACT_OUTPUT_DIR, bronzes)

    summary = pd.DataFrame(stats)
    print(f'TARGET_DATE = {TARGET_DATE} | SCALE_FACTOR = {SCALE_FACTOR}')
    print(f'TOTAL: {summary["rows"].sum():,} rows in {time.time() - _t0:.1f}s -> {FACT_OUTPUT_DIR}')
    print('Next: run autoloader.ipynb to ingest into car_workshop.fact.*')

In [0]:
%python
# Example: Loop through all dates and generate data for each
import time
import pandas as pd

all_stats = []
total_start = time.time()

for i, target_date in enumerate(date_list, 1):
    print(f'\n[{i}/{len(date_list)}] Processing {target_date}...')
    
    day_start = time.time()
    stats = generate_day(target_date, SCALE_FACTOR, FACT_OUTPUT_DIR, bronzes)
    day_elapsed = time.time() - day_start
    
    summary = pd.DataFrame(stats)
    total_rows = summary['rows'].sum()
    all_stats.extend(stats)
    
    print(f'  ✓ {total_rows:,} rows in {day_elapsed:.1f}s')

total_elapsed = time.time() - total_start
final_summary = pd.DataFrame(all_stats)
print(f'\n=== COMPLETE ===')
print(f'Dates: {len(date_list)} days ({date_list[0]} to {date_list[-1]})')
print(f'Total rows: {final_summary["rows"].sum():,}')
print(f'Total time: {total_elapsed:.1f}s ({total_elapsed/60:.1f} minutes)')
print(f'\nNext: run autoloader.ipynb to ingest into car_workshop.bronze.*')